• Вдохновившись примером великого Владимира Набокова, Ваши друзья занялись
лепидоптерологией (изучением бабочек). Часто при возвращении из поездки с новыми
образцами им бывает очень трудно определить, сколько же разных видов было
поймано, ведь многие виды очень похожи друг на друга.
• Вашим друзьям хотелось бы разделить n образцов на две группы (относящиеся к видам
A и B), но напрямую классифицировать каждый отдельно взятый образец очень трудно.
Вместо этого они решают действовать немного иначе.
• Каждая пара образцов i и j располагается рядом и тщательно изучается. Если
исследователи достаточно уверены в своей оценке, они помечают образцы из пары (i, j)
либо как «одинаковые» (то есть как относящиеся к одному виду), либо как «разные» (то
есть как относящиеся к разным видам). Также они могут отказаться от назначения
оценки конкретной паре; в таком случае пара называется неопределенной. Итак,
имеется набор из n образцов, а также набор из m оценок («одинаковые» или «разные»)
для пар, которые не были отнесены к неопределенным.
• Вашим друзьям хотелось бы знать, соответствуют ли эти данные представлению о том,
что каждая бабочка относится к одному из видов A или B. Говоря конкретнее, набор из
m оценок объявляется непротиворечивым, если возможно пометить каждый образец A
или B так, чтобы для каждой пары (i, j) с пометкой «одинаковые» i и j относились к
одному виду, а для каждой пары (i, j) с пометкой «разные» i и j относились к разным
видам. Ваши друзья возятся с проверкой непротиворечивости оценок, как вдруг один из
них осознает: ведь вы сможете придумать алгоритм, который позволит с ходу получить
ответ на этот вопрос.
• Предложите алгоритм для проверки непротиворечивости оценок с временем
выполнения O(m + n). 

1. DSU - это структура данных для работы с множеством непересекающихся множеств.
Есть элементы, которые делятся на группы, нужно быстро:
1) Определить, к какой группе принадлежит элемент
2) Объединить две группы в одну

Каждый образец - это элемент. Если два образца помечены как "одинаковые", нужно объеденить их в одну группу (один вид). Если два образца помечены как "разные", нужно убедиться, что они не находятся в одной группу.

Каждая операция (find или union) работает почти за O(1).

1) "Одинаковые", значит нужно объединить элементы в одну группу (union)
2) "Разные", следовательно проверяем, что элементы из разных групп (find(x) != find(y)). Если они уже в одной группу, противоречие.
3) После объединения всех "одинаковых" можно строить граф для проверки двудольности по "разным" связям.

In [6]:
def make_dsu(n):
    p = list(range(n))
    r = [0] * n
    return p, r

def find(p, x):
    if p[x] != x:
        p[x] = find(p, p[x])
    return p[x]

def union(p, r, a, b):
    a, b = find(p, a), find(p, b)
    if a == b:
        return
    if r[a] < r[b]:
        a, b = b, a
    p[b] = a
    if r[a] == r[b]:
        r[a] += 1

2. Построение. 

Нужно:
1) Объединить все элементы, которые точно "одинаковые".

Используем DSU, чтобы все элементы одной группы имели общий корень. Каждая компонента теперь представляет один вид.

2) Проверить пары "разные" на противоречия.

Для каждой пары "differend" проверяем:
- Если элементы уже в одной компоненте, следовательно противоречие (возвращаем None).
- Иначе добавляем ребро между корнями компонент.
Теперь граф описывает только требования "разные" между компонентами.

3) Построить граф компонент, чтобы потом проверить двудольность.

p - родитель каждого элемента (DSU)
r - ранги DSU
g - граф компонент для последующей проверки двудольности

In [7]:
def build(n, constraints):
    p, r = make_dsu(n)

    for i, j, t in constraints:
        if t == "same":
            union(p, r, i, j)

    g = [[] for _ in range(n)]

    for i, j, t in constraints:
        if t == "different":
            u, v = find(p, i), find(p, j)
            if u == v:
                return None, None, None
            g[u].append(v)
            g[v].append(u)

    return p, r, g

3. Проверка. 
Нужно проверить, можно ли раскрасить все компоненты бабочек в два вида (A и B), соблюдая ограничения "разные" (проверяем двудольность графа).

Идея:

1) Инициализация: создаем массив color, где -1 = компоненту еще не раскрашивали, 0 и 1 - два вида.
2) Обход всех элементов: для каждого элемента i находим его компоненту v = find(p, i). Если корень уже раскрашен, пропускаем, иначе начинаем обход.
3) BFS по компоненте: ставим корню цвет 0, помещаем в очередь. Пока очередь не пуста:
- Берем вершину x, смотрим всех соседей у y в графе "разные".
- Если сосед еще не раскрашен, красим в противоположный цвет 1 - color[x] и добавляем в очередь.
- Если сосед уже раскрашен в тот же цвет, следовательно противоречие, значит возращаем False.
4) Если обход завершен без конфликтов, значит возращаем True.

BFS = O(n + m), где n (вершины) и m (ребра "разные")

In [8]:
from collections import deque

def check(n, p, g):
    color = [-1] * n

    for i in range(n):
        v = find(p, i)
        if color[v] != -1:
            continue

        q = deque([v])
        color[v] = 0

        while q:
            x = q.popleft()
            for y in g[x]:
                if color[y] == -1:
                    color[y] = 1 - color[x]
                    q.append(y)
                elif color[y] == color[x]:
                    return False

    return True

4. Интерфейс

Вызываем build, если build обнаружил конфликт, значит возвращаем False. Иначе вызваем check, чтобы проверить, можно ли раскрасить граф компонент в два цвета (виды A и B)

In [ ]:
def is_consistent(n, constraints):
    p, r, g = build(n, constraints)
    if p is None:
        return False
    return check(n, p, g)

5. Пример

In [5]:
n = 4
constraints = [
    (0, 1, "same"),
    (1, 2, "different"),
    (2, 3, "same"),
    (0, 3, "different")
]

print("YES" if is_consistent(n, constraints) else "NO")

YES
